# Results visualization

Four market-design cases × five risk settings (`RN` + `RA beta 0.2/0.4/0.6/0.8`).

Reads CSVs from **`Results/`**. Missing runs (for example the green social planners) are skipped.

**This notebook is a clean rebuild.** Run the data cells first. Plots are added one at a time below the design system — none ship until we choose them.

---

## House style — *North Sea*

Default Matplotlib/Seaborn charts read as templates: DejaVu, `tab10` rainbow, boxed legends, grey plot faces, rotated y-labels, centered titles. Editorial graphics (FT, *Economist*, *Graphic Detail*) share a different grammar:

1. **Type hierarchy, not decoration.** A serif display line for the claim; a humanist sans for numbers and axes. Here: **Sitka** (titles) + **Segoe UI** (data). Both are already on Windows.
2. **The title is the takeaway.** Left-aligned. A short kicker above, a muted subtitle below that states units and coverage. The chart does not repeat the title.
3. **Paper, not dashboard.** Warm uncoated stock (`#F4F0E8`), warm black ink. No seaborn grey pane, no 3-D, no drop shadows, no rainbow.
4. **One hero colour.** North-sea teal for the series that matters; sand / copper / dusk as supporting voices. Risk aversion runs **calm → heat** (teal → copper), not five unrelated hues.
5. **Chrome instead of chartjunk.** A thin teal rule at the top; hairline baseline; horizontal grid only; no top/right spines; no legend box. Prefer **direct labels**.
6. **Y-unit as a header**, sitting above the axis in sentence case — not a rotated side label.
7. **Export like a print desk.** PNG @ 300 dpi plus PDF and SVG with embedded TrueType (`pdf.fonttype = 42`).

Colourblind-safe enough for print: teal vs copper vs dusk vs sand are separable in grayscale by lightness, not only hue.


In [ ]:
# Imports
from __future__ import annotations

import os
import textwrap
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
import yaml
from matplotlib import font_manager as fm
from matplotlib.lines import Line2D
from matplotlib.ticker import MaxNLocator, PercentFormatter

print("Imports OK")


In [ ]:
# Paths and case catalog
def find_project_root() -> Path:
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "Data" / "data.yaml").is_file():
            return p
    return here

ROOT = find_project_root()
RESULTS_ROOT = ROOT / "Results"
if not RESULTS_ROOT.is_dir():
    alt = sorted(
        (p for p in ROOT.glob("Results*") if p.is_dir()),
        key=lambda p: p.stat().st_mtime,
        reverse=True,
    )
    RESULTS_ROOT = alt[0] if alt else RESULTS_ROOT

FIG_DIR = ROOT / "visualization_figures"
FIG_DIR.mkdir(exist_ok=True)
TABLE_DIR = FIG_DIR / "tables"
TABLE_DIR.mkdir(exist_ok=True)

RISK_SPECS = [
    ("RN", "RN", ("RN", "Risk Neutral", "risk_neutral")),
    ("RA beta 0.2", "0.2", ("RA beta 0.2", "Beta 0.2", "β=0.2", "beta_0.2")),
    ("RA beta 0.4", "0.4", ("RA beta 0.4", "Beta 0.4", "β=0.4", "beta_0.4")),
    ("RA beta 0.6", "0.6", ("RA beta 0.6", "Beta 0.6", "β=0.6", "beta_0.6")),
    ("RA beta 0.8", "0.8", ("RA beta 0.8", "Beta 0.8", "β=0.8", "beta_0.8")),
]
RISK_ORDER = [key for key, _, _ in RISK_SPECS]
RISK_FOLDERS = {key: label for key, label, _ in RISK_SPECS}
RISK_LABELS = [RISK_FOLDERS[k] for k in RISK_ORDER]

RISK_DIRS: dict[str, Path] = {}
for key, _label, aliases in RISK_SPECS:
    for alias in aliases:
        p = RESULTS_ROOT / alias
        if p.is_dir():
            RISK_DIRS[key] = p
            break
    else:
        RISK_DIRS[key] = RESULTS_ROOT / key

CASE_FOLDERS = {
    "sp": "social_planner_results",
    "me": "market_exposure_results",
    "gsp": "green_social_planner_results",
    "gh2": "green_h2_social_planner_results",
}
CASE_TITLES = {
    "sp": "Social planner",
    "me": "Market exposure",
    "gsp": "Green social planner",
    "gh2": "Green H2 social planner",
}
CASE_SHORT = {"sp": "SP", "me": "ME", "gsp": "G-SP", "gh2": "GH2-SP"}

PRICE_COLS = {
    "elec": "Elec_Price",
    "H2": "H2_Price",
    "elec_GC": "Elec_GC_Price",
    "H2_GC": "H2_GC_Price",
    "EP": "EP_Price",
}
MARKET_LABELS = {
    "elec": "Electricity",
    "H2": "Hydrogen",
    "elec_GC": "Electricity GC",
    "H2_GC": "H2 GC",
    "EP": "Ammonia",
}

INVESTABLE = [
    "Gen_VRES_Solar",
    "Gen_VRES_Wind",
    "Prod_H2_Green",
    "Offtaker_Green",
]
INVESTABLE_LABELS = {
    "Gen_VRES_Solar": "Solar",
    "Gen_VRES_Wind": "Wind",
    "Prod_H2_Green": "Electrolyzer",
    "Offtaker_Green": "Green EP plant",
}
INVESTABLE_UNITS = {
    "Gen_VRES_Solar": "GW",
    "Gen_VRES_Wind": "GW",
    "Prod_H2_Green": "MW",
    "Offtaker_Green": "MW",
}
INVESTABLE_SCALE = {
    "Gen_VRES_Solar": 1e-3,
    "Gen_VRES_Wind": 1e-3,
    "Prod_H2_Green": 1.0,
    "Offtaker_Green": 1.0,
}

N_TS, N_RD, N_YR = 24, 8, 15

print(f"ROOT         = {ROOT}")
print(f"RESULTS_ROOT = {RESULTS_ROOT}   exists={RESULTS_ROOT.is_dir()}")
print(f"FIG_DIR      = {FIG_DIR}")
print("Risk folders:")
for key in RISK_ORDER:
    p = RISK_DIRS[key]
    flag = "OK" if p.is_dir() else "MISSING"
    print(f"  {key:14s} -> {p.name if p.exists() else '(missing)':20s}  {flag}")
print("Case subfolders:")
for case_key, folder in CASE_FOLDERS.items():
    found, missing = [], []
    for key in RISK_ORDER:
        d = RISK_DIRS[key] / folder
        (found if d.is_dir() else missing).append(key)
    status = ", ".join(found) if found else "none"
    extra = f"  (missing: {', '.join(missing)})" if missing else ""
    print(f"  {case_key:4s} {folder:36s} {status}{extra}")


In [ ]:
# Design system — North Sea
WIN_FONTS = Path(os.environ.get("WINDIR", r"C:\Windows")) / "Fonts"

def _font(*filenames: str, family: str = "sans-serif") -> fm.FontProperties:
    for name in filenames:
        path = WIN_FONTS / name
        if path.is_file():
            try:
                fm.fontManager.addfont(str(path))
            except (OSError, RuntimeError, ValueError):
                pass
            return fm.FontProperties(fname=str(path))
    return fm.FontProperties(family=family)

FONT_SANS = _font("segoeui.ttf")
FONT_SANS_SB = _font("seguisb.ttf", "segoeui.ttf")
FONT_SANS_BD = _font("segoeuib.ttf", "seguisb.ttf", "segoeui.ttf")
FONT_SERIF = _font("SitkaVF.ttf", "georgia.ttf", "constan.ttf", family="serif")
FONT_SERIF_BD = _font("georgiab.ttf", "SitkaVF.ttf", "georgia.ttf", family="serif")

SANS_NAME = FONT_SANS.get_name()
SERIF_NAME = FONT_SERIF.get_name()

# Surfaces
PAPER = "#F4F0E8"
INK = "#1A1714"
INK_SOFT = "#5C574E"
RULE = "#D4CDBF"
GRID = "#E4DDD0"
# Hero + supporting
TEAL = "#0E6B66"
TEAL_DEEP = "#0A3F3C"
SEA = "#2F8A84"
FOAM = "#8BBDB6"
INDIGO = "#1E3A5F"
DUSK = "#5A7A96"
SAND = "#C4A36A"
COPPER = "#B85C38"
CLAY = "#8C3D2B"
MIST = "#A39C90"
REF = "#8A847A"

RISK_COLORS = {
    "RN": TEAL,
    "RA beta 0.2": SEA,
    "RA beta 0.4": DUSK,
    "RA beta 0.6": SAND,
    "RA beta 0.8": COPPER,
}
CASE_COLORS = {
    "sp": INDIGO,
    "me": TEAL,
    "gsp": COPPER,
    "gh2": SAND,
}
MARKET_COLORS = {
    "elec": INDIGO,
    "H2": TEAL,
    "elec_GC": DUSK,
    "H2_GC": SAND,
    "EP": COPPER,
}
INV_COLORS = {
    "Gen_VRES_Solar": SAND,
    "Gen_VRES_Wind": INDIGO,
    "Prod_H2_Green": TEAL,
    "Offtaker_Green": COPPER,
}

plt.rcParams.update({
    "figure.dpi": 140,
    "savefig.dpi": 300,
    "figure.facecolor": PAPER,
    "axes.facecolor": PAPER,
    "axes.edgecolor": RULE,
    "axes.labelcolor": INK,
    "axes.titlecolor": INK,
    "axes.titlesize": 14,
    "axes.labelsize": 10.5,
    "axes.titlelocation": "left",
    "axes.titleweight": "regular",
    "axes.grid": False,
    "axes.axisbelow": True,
    "axes.unicode_minus": False,
    "axes.linewidth": 0.7,
    "font.size": 10.5,
    "font.family": SANS_NAME,
    "text.color": INK,
    "xtick.color": INK_SOFT,
    "ytick.color": INK_SOFT,
    "xtick.direction": "out",
    "ytick.direction": "out",
    "xtick.major.size": 0,
    "ytick.major.size": 0,
    "legend.frameon": False,
    "legend.fontsize": 9.5,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",
    "savefig.facecolor": PAPER,
    "savefig.edgecolor": "none",
    "lines.solid_capstyle": "round",
    "lines.solid_joinstyle": "round",
})


def risk_xlabels() -> list[str]:
    return list(RISK_LABELS)


def _finite(val) -> bool:
    try:
        return bool(np.isfinite(float(val)))
    except (TypeError, ValueError):
        return False


def fmt_num(v: float, digits: int = 1) -> str:
    if not _finite(v):
        return "—"
    av = abs(float(v))
    if av >= 100:
        return f"{v:.0f}"
    if av >= 10:
        return f"{v:.1f}"
    if av >= 1:
        return f"{v:.{digits}f}"
    return f"{v:.2f}"


def new_figure(width: float = 8.6, height: float = 4.8):
    fig, ax = plt.subplots(figsize=(width, height))
    fig.patch.set_facecolor(PAPER)
    ax.set_facecolor(PAPER)
    fig.subplots_adjust(left=0.09, right=0.97, bottom=0.14, top=0.86)
    return fig, ax


def finish_ax(ax, ylabel: str | None = None):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_visible(False)
    ax.spines["bottom"].set_color(RULE)
    ax.spines["bottom"].set_linewidth(0.9)
    ax.tick_params(axis="both", length=0, labelsize=9.5, colors=INK_SOFT)
    ax.yaxis.grid(True, color=GRID, linewidth=0.7, zorder=0)
    ax.xaxis.grid(False)
    ax.set_axisbelow(True)
    ax.yaxis.set_major_locator(MaxNLocator(nbins=5, min_n_ticks=3))
    if ylabel:
        ax.set_title(ylabel, loc="left", fontproperties=FONT_SANS, fontsize=10, color=INK_SOFT, pad=8)


def add_chrome(fig, *, title: str, **_kwargs):
    fig.text(
        0.08, 0.97, title,
        fontproperties=FONT_SERIF, fontsize=13.5, color=INK,
        ha="left", va="top",
    )


def save_figure(fig, stem: str, table: pd.DataFrame | None = None) -> list[Path]:
    written = []
    for ext in ("png", "svg", "pdf"):
        path = FIG_DIR / f"{stem}.{ext}"
        try:
            fig.savefig(path, bbox_inches="tight", pad_inches=0.28, facecolor=PAPER)
            written.append(path)
        except (ValueError, RuntimeError, OSError) as exc:
            if ext == "pdf":
                print(f"PDF skipped for {stem} ({type(exc).__name__})")
                continue
            raise
    if table is not None:
        csv = TABLE_DIR / f"{stem}.csv"
        table.to_csv(csv, index=False, float_format="%.6g")
        written.append(csv)
        print(f"saved {stem}.png / .svg  +  tables/{csv.name}")
    else:
        print(f"saved {stem}.png / .svg")
    return written


print(f"Type   display={SERIF_NAME}   data={SANS_NAME}")
print("North Sea design system ready — no figures yet.")


In [ ]:
# YAML helpers (grey ammonia MC + capacity seeds)
with open(ROOT / "Data" / "data.yaml", encoding="utf-8") as f:
    DATA_YAML = yaml.safe_load(f)

fuel = DATA_YAML["Fuel"]
grey = DATA_YAML["Hydrogen_Offtaker"]["Offtaker_Grey"]
gas_mults = [float(x) for x in DATA_YAML["Scenarios"]["gas_price_multipliers"]]
weather_years = [int(x) for x in DATA_YAML["Scenarios"]["weather_years"]]
EP_TOTAL_DEMAND = float(DATA_YAML["EP_market"]["Total_Demand"])
EP_DEMAND_COL = str(DATA_YAML["EP_market"].get("Demand_Column", "LOAD_EP"))

CAPACITY_SEED = {
    "Gen_VRES_Solar": float(DATA_YAML["Power"]["Gen_VRES_Solar"]["Capacity"]),
    "Gen_VRES_Wind": float(DATA_YAML["Power"]["Gen_VRES_Wind"]["Capacity"]),
    "Prod_H2_Green": float(DATA_YAML["Hydrogen"]["Prod_H2_Green"]["Capacity_H2_Output"]),
    "Offtaker_Green": float(DATA_YAML["Hydrogen_Offtaker"]["Offtaker_Green"]["Capacity_EP_Out"]),
}

def grey_ammonia_mc(gas_multiplier: float) -> float:
    return (
        float(grey["GasIntensity"]) * float(fuel["GasPrice"]) * float(gas_multiplier)
        + float(grey["CO2Intensity"]) * float(fuel["CO2Price"])
        + float(grey["VariableOM"])
    )

GREY_MC_BY_GAS = {g: grey_ammonia_mc(g) for g in gas_mults}
GREY_MC_EXPECTED = float(np.mean(list(GREY_MC_BY_GAS.values())))

N_TS = int(DATA_YAML["General"].get("nTimesteps", N_TS))
N_RD = int(DATA_YAML["General"].get("nReprDays", N_RD))
N_YR = len(weather_years) * len(gas_mults)

print("Gas multipliers:", gas_mults)
print(f"Expected grey MC: {GREY_MC_EXPECTED:.2f} EUR/MWh_EP")
print("Capacity seeds (MW):", {k: round(v, 1) for k, v in CAPACITY_SEED.items()})
print(f"Time grid: {N_TS} x {N_RD} x {N_YR} = {N_TS * N_RD * N_YR} slots")


In [ ]:
# CSV loaders
REQUIRED_CSVS = (
    "Market_Prices.csv",
    "Agent_Objectives_Per_Timestep.csv",
    "Agent_Summary.csv",
)

def run_dir(risk_key: str, case_key: str) -> Path:
    return RISK_DIRS[risk_key] / CASE_FOLDERS[case_key]

def load_metrics_csv(path: Path) -> dict:
    df = pd.read_csv(path)
    out = {}
    if "Metric" not in df.columns or "Value" not in df.columns:
        return out
    for _, row in df.iterrows():
        key = str(row["Metric"])
        val = row["Value"]
        try:
            out[key] = float(val)
        except (TypeError, ValueError):
            out[key] = val
    return out

def _system_cost_from_welfare(case_dir: Path) -> float:
    wpath = case_dir / "Welfare_By_Agent_Per_Year.csv"
    if not wpath.is_file():
        return np.nan
    wdf = pd.read_csv(wpath)
    if "welfare" not in wdf.columns or "probability" not in wdf.columns:
        return np.nan
    if "is_demand" in wdf.columns:
        demand = wdf["is_demand"].astype(str).str.lower().isin(("true", "1", "yes"))
        rest = wdf.loc[~demand]
    else:
        rest = wdf
    return float((-rest["welfare"].astype(float) * rest["probability"].astype(float)).sum())

def load_run_metrics(case_dir: Path) -> dict:
    out: dict = {}
    for name in ("System_Metrics.csv", "Risk_Metrics.csv"):
        p = case_dir / name
        if p.is_file():
            out.update(load_metrics_csv(p))
    if not _finite(out.get("expected_total_system_cost")):
        if _finite(out.get("system_cost_expected")):
            out["expected_total_system_cost"] = float(out["system_cost_expected"])
        elif _finite(out.get("expected_welfare_ex_demand")):
            out["expected_total_system_cost"] = -float(out["expected_welfare_ex_demand"])
        else:
            out["expected_total_system_cost"] = _system_cost_from_welfare(case_dir)
    return out

def load_prices_with_weights(case_dir: Path) -> pd.DataFrame:
    prices = pd.read_csv(case_dir / "Market_Prices.csv")
    ao = pd.read_csv(
        case_dir / "Agent_Objectives_Per_Timestep.csv",
        usecols=["Time", "jh", "jd", "jy", "W"],
    )
    df = prices.merge(ao, on="Time", how="left", validate="one_to_one")
    if df["W"].isna().any():
        raise ValueError(f"Missing W after merge in {case_dir}")
    return df

def slot_weights(df: pd.DataFrame) -> np.ndarray:
    w = df["W"].to_numpy(dtype=float)
    n_scen = int(df["jy"].nunique()) if "jy" in df.columns else N_YR
    n_scen = n_scen if n_scen > 0 else N_YR
    return w * (1.0 / n_scen)

def w_mean_price(df: pd.DataFrame, col: str) -> float:
    if col not in df.columns:
        return np.nan
    y = df[col].to_numpy(dtype=float)
    w = df["W"].to_numpy(dtype=float)
    if not np.any(np.isfinite(w)) or float(np.nansum(w)) == 0.0:
        return float(np.nanmean(y))
    mask = np.isfinite(y) & np.isfinite(w)
    if not np.any(mask):
        return np.nan
    return float(np.average(y[mask], weights=w[mask]))

def cwap_from_qty_price(qty: np.ndarray, price: np.ndarray, wt: np.ndarray) -> float:
    q = np.maximum(np.asarray(qty, dtype=float), 0.0)
    p = np.asarray(price, dtype=float)
    w = np.asarray(wt, dtype=float)
    mask = np.isfinite(q) & np.isfinite(p) & np.isfinite(w)
    den = float(np.sum(w[mask] * q[mask]))
    if den <= 1e-12:
        return np.nan
    return float(np.sum(w[mask] * p[mask] * q[mask]) / den)

def ep_qty_vector(ao: pd.DataFrame) -> np.ndarray:
    cols = [c for c in ("Offtaker_Green_ep", "Offtaker_Grey_ep", "Offtaker_Import_ep") if c in ao.columns]
    if not cols:
        return np.full(len(ao), np.nan)
    return ao[cols].to_numpy(dtype=float).sum(axis=1)

SLOT_TO_AGENT = {
    "VRES_solar": "Gen_VRES_Solar",
    "VRES_wind": "Gen_VRES_Wind",
    "H2": "Prod_H2_Green",
    "EP": "Offtaker_Green",
}

def load_investments(case_dir: Path) -> pd.Series:
    inv = pd.Series(np.nan, index=INVESTABLE, dtype=float)
    inv_path = case_dir / "Investable_Capacities.csv"
    summary_path = case_dir / "Agent_Summary.csv"
    summary = pd.read_csv(summary_path).set_index("AgentID") if summary_path.is_file() else None

    if inv_path.exists():
        df = pd.read_csv(inv_path).set_index("AgentID")
        for aid in INVESTABLE:
            if aid in df.index:
                inv[aid] = float(df.loc[aid, "Investment_Total_MW"])

    if summary is not None:
        for aid in INVESTABLE:
            if pd.notna(inv[aid]):
                continue
            if aid not in summary.index:
                continue
            v = float(summary.loc[aid, "Investment_Total_MW"])
            cap = float(summary.loc[aid, "Capacity_Final_MW"])
            inv[aid] = v if v > 0 else max(0.0, cap - CAPACITY_SEED[aid])

    merged_path = case_dir / "Merged_Capacities.csv"
    if merged_path.is_file():
        mdf = pd.read_csv(merged_path)
        for _, row in mdf.iterrows():
            aid = SLOT_TO_AGENT.get(str(row["Slot"]))
            if aid is None:
                continue
            v = float(row["Investment_MW"])
            cap = float(row["Capacity_MW"])
            inv[aid] = v if v > 0 else max(0.0, cap - CAPACITY_SEED[aid])

    sp_cap_path = case_dir / "SP_Capacities.csv"
    if sp_cap_path.is_file():
        spc = pd.read_csv(sp_cap_path)
        for aid in INVESTABLE:
            if pd.notna(inv[aid]):
                continue
            hit = spc.loc[spc["AgentID"] == aid, "cap"] if "AgentID" in spc.columns else pd.Series(dtype=float)
            if hit.empty:
                continue
            cap = float(hit.mean())
            inv[aid] = max(0.0, cap - CAPACITY_SEED[aid])

    return inv.fillna(0.0)

def metrics_row(sm: dict, prices_df: pd.DataFrame, ao: pd.DataFrame) -> dict:
    wt = slot_weights(prices_df)
    q_elec = ao["Cons_Elec_01_d"].to_numpy(dtype=float) if "Cons_Elec_01_d" in ao.columns else None
    q_gc = (
        ao["Demand_GC_Elec_01_d_gc"].to_numpy(dtype=float)
        if "Demand_GC_Elec_01_d_gc" in ao.columns
        else None
    )
    if "D_EP_FLAT" in globals() and len(D_EP_FLAT) == len(prices_df):
        q_ep = D_EP_FLAT
    else:
        q_ep = ep_qty_vector(ao)

    cwap_elec = sm.get("consumer_Cons_Elec_01_elec_weighted_avg_price", np.nan)
    if not _finite(cwap_elec) and q_elec is not None:
        cwap_elec = cwap_from_qty_price(q_elec, prices_df["Elec_Price"], wt)

    cwap_gc = sm.get("consumer_Demand_GC_Elec_01_elec_GC_weighted_avg_price", np.nan)
    if not _finite(cwap_gc) and q_gc is not None:
        cwap_gc = cwap_from_qty_price(q_gc, prices_df["Elec_GC_Price"], wt)

    cwap_ep = sm.get("consumer_Ammonia_Demand_EP_weighted_avg_price", sm.get("ep_cwap", np.nan))
    if not _finite(cwap_ep):
        cwap_ep = cwap_from_qty_price(q_ep, prices_df["EP_Price"], wt)

    cost = sm.get("expected_total_system_cost", np.nan)
    return {
        "gamma": sm.get("gamma", np.nan),
        "beta": sm.get("beta", np.nan),
        "system_cost_bn": (float(cost) / 1e9) if _finite(cost) else np.nan,
        "cwap_elec": cwap_elec,
        "cwap_elec_GC": cwap_gc,
        "cwap_EP": cwap_ep,
        **{f"wmean_{k}": w_mean_price(prices_df, c) for k, c in PRICE_COLS.items()},
    }

print("Loader helpers defined")


In [ ]:
# Inelastic EP demand on the ADMM / SP time grid
def build_ep_demand_mwh() -> np.ndarray:
    n_gas = len(gas_mults)
    out = np.zeros(N_TS * N_RD * N_YR, dtype=float)
    idx = 0
    for weather in weather_years:
        ts = pd.read_csv(ROOT / "Input" / f"timeseries_{weather}.csv")
        profile = ts[EP_DEMAND_COL].to_numpy(dtype=float)
        for _g in range(n_gas):
            for jd in range(N_RD):
                for jh in range(N_TS):
                    row = jd * N_TS + jh
                    out[idx] = EP_TOTAL_DEMAND * float(profile[row])
                    idx += 1
    return out

D_EP_FLAT = build_ep_demand_mwh()
print(f"D_EP flat length={len(D_EP_FLAT)}, mean={D_EP_FLAT.mean():.1f} MW_EP")


In [ ]:
# Load every available run
system_metrics: dict[str, dict[str, dict]] = {ck: {} for ck in CASE_FOLDERS}
prices: dict[str, dict[str, pd.DataFrame]] = {ck: {} for ck in CASE_FOLDERS}
ao_full: dict[str, dict[str, pd.DataFrame]] = {ck: {} for ck in CASE_FOLDERS}
investments: dict[str, dict[str, pd.Series]] = {ck: {} for ck in CASE_FOLDERS}
summary: dict[str, pd.DataFrame] = {}
inv_tables: dict[str, pd.DataFrame] = {}
skipped_runs: list[str] = []

for case_key in CASE_FOLDERS:
    rows, inv_rows = [], []
    for risk in RISK_ORDER:
        d = run_dir(risk, case_key)
        if not d.is_dir():
            skipped_runs.append(f"{case_key}/{risk}: folder missing ({d.name})")
            continue
        missing = [name for name in REQUIRED_CSVS if not (d / name).is_file()]
        if missing:
            skipped_runs.append(f"{case_key}/{risk}: missing {', '.join(missing)}")
            continue
        try:
            sm = load_run_metrics(d)
            pr = load_prices_with_weights(d)
            ao = pd.read_csv(d / "Agent_Objectives_Per_Timestep.csv")
            inv = load_investments(d)
        except Exception as exc:
            skipped_runs.append(f"{case_key}/{risk}: {type(exc).__name__}: {exc}")
            continue

        system_metrics[case_key][risk] = sm
        prices[case_key][risk] = pr
        ao_full[case_key][risk] = ao
        investments[case_key][risk] = inv

        row = metrics_row(sm, pr, ao)
        row.update(risk_folder=risk, risk_label=RISK_FOLDERS[risk])
        rows.append(row)

        inv_row = {"risk_folder": risk, "risk_label": RISK_FOLDERS[risk]}
        for aid in INVESTABLE:
            inv_row[aid] = float(inv[aid])
        inv_rows.append(inv_row)

        cost_s = f"{row['system_cost_bn']:.3f} bn" if _finite(row["system_cost_bn"]) else "n/a"
        ep_s = f"{row['cwap_EP']:.1f}" if _finite(row["cwap_EP"]) else "n/a"
        print(f"loaded {case_key:4s} | {risk:14s} | cost={cost_s} | EP CWAP={ep_s}")

    if rows:
        summary[case_key] = pd.DataFrame(rows).set_index("risk_folder")
        inv_tables[case_key] = pd.DataFrame(inv_rows).set_index("risk_folder")
    else:
        print(f"no runs loaded for {case_key} ({CASE_TITLES[case_key]})")

AVAILABLE_CASES = [ck for ck in CASE_FOLDERS if ck in summary]
print(f"\nLoaded cases: {', '.join(AVAILABLE_CASES) if AVAILABLE_CASES else 'none'}")
if skipped_runs:
    print(f"Skipped {len(skipped_runs)} run(s):")
    for line in skipped_runs:
        print(f"  - {line}")
print("Done loading.")


In [ ]:
# Derived tables used by later figures
def scenario_probability(jy: np.ndarray) -> np.ndarray:
    return np.full(jy.shape, 1.0 / N_YR, dtype=float)

def compute_tracc(case_key: str, risk: str) -> dict:
    ao = ao_full[case_key][risk]
    pr = prices[case_key][risk]
    df = ao[["Time", "jh", "jd", "jy", "W", "Cons_Elec_01_d"]].merge(
        pr[["Time", "Elec_Price", "EP_Price"]], on="Time", validate="one_to_one"
    )
    if len(df) != len(D_EP_FLAT):
        raise ValueError(f"Time-grid mismatch for {case_key}/{risk}")

    wt = df["W"].to_numpy(float) * scenario_probability(df["jy"].to_numpy(int))
    q_elec = np.maximum(df["Cons_Elec_01_d"].to_numpy(float), 0.0)
    q_ep = np.maximum(D_EP_FLAT, 0.0)
    lam_elec = df["Elec_Price"].to_numpy(float)
    lam_ep = df["EP_Price"].to_numpy(float)

    exp_elec = float(np.sum(wt * lam_elec * q_elec))
    exp_ep = float(np.sum(wt * lam_ep * q_ep))
    qty_elec = float(np.sum(wt * q_elec))
    qty_ep = float(np.sum(wt * q_ep))
    exp_tot = exp_elec + exp_ep
    qty_tot = qty_elec + qty_ep
    return {
        "exp_elec": exp_elec,
        "exp_ep": exp_ep,
        "exp_total": exp_tot,
        "qty_elec": qty_elec,
        "qty_ep": qty_ep,
        "qty_total": qty_tot,
        "cwap_elec": exp_elec / qty_elec if qty_elec else np.nan,
        "cwap_ep": exp_ep / qty_ep if qty_ep else np.nan,
        "tracc": exp_tot / qty_tot if qty_tot else np.nan,
        "share_exp_elec": exp_elec / exp_tot if exp_tot else np.nan,
        "share_exp_ep": exp_ep / exp_tot if exp_tot else np.nan,
    }

tracc: dict[str, pd.DataFrame] = {}
for case_key in CASE_FOLDERS:
    rows = []
    for risk in RISK_ORDER:
        if risk not in ao_full.get(case_key, {}) or risk not in prices.get(case_key, {}):
            continue
        try:
            r = compute_tracc(case_key, risk)
        except Exception as exc:
            print(f"TRACC skip {case_key}/{risk}: {type(exc).__name__}: {exc}")
            continue
        r.update(risk_folder=risk, risk_label=RISK_FOLDERS[risk])
        rows.append(r)
        print(f"TRACC {case_key:4s} | {risk:14s} | {r['tracc']:.2f} EUR/MWh")
    if rows:
        tracc[case_key] = pd.DataFrame(rows).set_index("risk_folder")
    else:
        print(f"TRACC skipped for {case_key} ({CASE_TITLES[case_key]}) — no loaded runs")


def case_loaded(case_key: str) -> bool:
    return case_key in summary and not summary[case_key].empty

def skip_missing_case(case_key: str, what: str = "figure") -> bool:
    if case_loaded(case_key):
        return False
    print(f"Skipping {what} for {CASE_TITLES.get(case_key, case_key)} — no results loaded.")
    return True

def series_for(df: pd.DataFrame | None, col: str) -> list[float]:
    if df is None or df.empty:
        return [np.nan] * len(RISK_ORDER)
    return [float(df.loc[r, col]) if r in df.index else np.nan for r in RISK_ORDER]

def loaded_risks(case_key: str) -> list[str]:
    if case_key not in prices:
        return []
    return [r for r in RISK_ORDER if r in prices[case_key]]

print("Derived tables ready.")


## Figures

Each block is one chart. Re-run the data cells above, then run a figure cell on its own.


In [ ]:
# Figure helpers — refresh investments, shared drawing tools
from matplotlib import patheffects as pe
from matplotlib.patches import Patch

CASE_ORDER = [ck for ck in ("sp", "me", "gsp", "gh2") if ck in inv_tables]
HALO = [pe.withStroke(linewidth=3.4, foreground=PAPER)]

AID_TO_SLOT = {aid: slot for slot, aid in SLOT_TO_AGENT.items()}


def _has_disaggregated_cap(case_dir: Path, aid: str) -> bool:
    summary_path = case_dir / "Agent_Summary.csv"
    if summary_path.is_file():
        ids = set(pd.read_csv(summary_path)["AgentID"].astype(str))
        if aid in ids:
            return True
    merged_path = case_dir / "Merged_Capacities.csv"
    if merged_path.is_file():
        slots = set(pd.read_csv(merged_path)["Slot"].astype(str))
        if AID_TO_SLOT.get(aid) in slots:
            return True
    return False


def refresh_investments() -> None:
    for case_key in list(investments):
        rows = []
        for risk in RISK_ORDER:
            if risk not in investments[case_key]:
                continue
            d = run_dir(risk, case_key)
            inv = load_investments(d)
            if case_key in ("gsp", "gh2") and "sp" in investments and risk in investments["sp"]:
                for aid in INVESTABLE:
                    if not _has_disaggregated_cap(d, aid):
                        inv[aid] = float(investments["sp"][risk][aid])
            investments[case_key][risk] = inv
            row = {"risk_folder": risk, "risk_label": RISK_FOLDERS[risk]}
            for aid in INVESTABLE:
                row[aid] = float(inv[aid])
            rows.append(row)
        if rows:
            inv_tables[case_key] = pd.DataFrame(rows).set_index("risk_folder")


refresh_investments()
print("Investments (MW, new capacity)")
for ck in CASE_ORDER:
    print(f"\n{CASE_SHORT[ck]}")
    print(inv_tables[ck][INVESTABLE].round(1).to_string())


def new_grid(nrows: int, ncols: int, *, width=10.8, height=4.8, top=0.84, bottom=0.16, wspace=0.28, hspace=0.42):
    fig, axes = plt.subplots(nrows, ncols, figsize=(width, height), squeeze=False)
    fig.patch.set_facecolor(PAPER)
    fig.subplots_adjust(left=0.07, right=0.98, bottom=bottom, top=top, wspace=wspace, hspace=hspace)
    for ax in axes.ravel():
        ax.set_facecolor(PAPER)
    return fig, axes


def inv_mw(case_key: str, risk: str, aid: str) -> float:
    if case_key not in inv_tables or risk not in inv_tables[case_key].index:
        return np.nan
    return float(inv_tables[case_key].loc[risk, aid])


def grouped_bars(ax, categories, series_keys, values, color_of):
    n_s = len(series_keys)
    x = np.arange(len(categories), dtype=float)
    width = min(0.15, 0.76 / max(n_s, 1))
    offsets = (np.arange(n_s) - (n_s - 1) / 2.0) * width
    all_vals = []
    for i, sk in enumerate(series_keys):
        vals = []
        for v in values[sk]:
            if not _finite(v) or abs(float(v)) < 1e-6:
                vals.append(np.nan)
            else:
                vals.append(float(v))
        all_vals.extend(vals)
        ax.bar(
            x + offsets[i], vals, width=width * 0.90,
            color=color_of(sk), zorder=3, linewidth=0, align="center",
        )
    ax.set_xticks(x)
    ax.set_xticklabels(categories, fontproperties=FONT_SANS, fontsize=8.5, color=INK_SOFT)
    finite = [v for v in all_vals if _finite(v)]
    hi = max(finite) if finite else 0.0
    ax.set_ylim(0, (hi * 1.18) if hi > 0 else 1.0)
    if hi <= 0:
        ax.text(
            0.5, 0.5, "No new capacity", transform=ax.transAxes,
            ha="center", va="center", fontproperties=FONT_SANS, fontsize=10, color=MIST,
        )
    return finite


def panel_unit(ax, text: str):
    ax.set_title(text, loc="left", fontproperties=FONT_SANS, fontsize=10, color=INK_SOFT, pad=8)


def nice_ylim(ax, values, *, min_span: float = 8.0):
    finite = [float(v) for v in values if _finite(v)]
    if not finite:
        return
    lo, hi = min(finite), max(finite)
    span = max(hi - lo, min_span)
    ax.set_ylim(hi - span * 1.12, hi + span * 0.18)


def fig_legend(fig, handles, labels, ncol: int):
    fig.legend(
        handles=list(handles),
        labels=list(labels),
        loc="lower center", ncol=ncol,
        bbox_to_anchor=(0.5, 0.02), frameon=False,
        prop=FONT_SANS, fontsize=8.5, handlelength=1.1, columnspacing=1.4,
    )


def price_duration(df: pd.DataFrame, col: str):
    p = df[col].to_numpy(dtype=float)
    w = df["W"].to_numpy(dtype=float)
    mask = np.isfinite(p) & np.isfinite(w)
    p, w = p[mask], w[mask]
    order = np.argsort(-p)
    p_sorted = p[order]
    w_sorted = w[order]
    cum = np.cumsum(w_sorted)
    return cum / cum[-1], p_sorted


def draw_price_duration(ax, case_key: str, market: str):
    col = PRICE_COLS[market]
    means = []
    for risk in loaded_risks(case_key):
        df = prices[case_key][risk]
        x, y = price_duration(df, col)
        ax.plot(x, y, color=RISK_COLORS[risk], lw=2.15, zorder=3, solid_capstyle="round")
        mu = w_mean_price(df, col)
        means.append((risk, mu, y[0]))
    finish_ax(ax)
    panel_unit(ax, "Duration curve  (EUR / MWh)")
    ax.set_xlim(0, 1)
    ax.xaxis.set_major_formatter(PercentFormatter(1.0))
    ax.set_xlabel("Share of weighted hours", fontproperties=FONT_SANS, fontsize=9.5, color=INK_SOFT)
    ax.axhline(0, color=RULE, lw=0.7, zorder=1)
    return means


def draw_mean_prices(ax, means, unit: str = "EUR / MWh"):
    by_risk = {r: m for r, m, _ in means}
    ys = [by_risk.get(r, np.nan) for r in RISK_ORDER]
    x = np.arange(len(RISK_ORDER))
    ax.bar(x, ys, color=[RISK_COLORS[r] for r in RISK_ORDER], width=0.62, zorder=3, linewidth=0)
    for i, v in enumerate(ys):
        if not _finite(v):
            continue
        ax.text(
            x[i], v, f"{v:.1f}", ha="center", va="bottom", fontsize=8.0,
            color=INK, fontproperties=FONT_SANS, clip_on=False,
        )
    ax.set_xticks(x, risk_xlabels())
    finish_ax(ax)
    panel_unit(ax, f"Mean price  ({unit})")
    nice_ylim(ax, ys, min_span=8.0)
    return ys


def plot_price_pair(case_key: str, market: str, *, title: str, stem: str, unit: str = "EUR / MWh"):
    fig, axes = new_grid(1, 2, width=10.6, height=4.6, top=0.84, bottom=0.18, wspace=0.30)
    ax_d, ax_m = axes[0, 0], axes[0, 1]
    means = draw_price_duration(ax_d, case_key, market)
    panel_unit(ax_d, f"Duration curve  ({unit})")
    ys = draw_mean_prices(ax_m, means, unit=unit)
    handles = [Line2D([0], [0], color=RISK_COLORS[r], lw=2.15) for r in RISK_ORDER]
    fig_legend(fig, handles, [RISK_FOLDERS[r] for r in RISK_ORDER], ncol=5)
    add_chrome(fig, title=title)
    save_figure(fig, stem, pd.DataFrame({
        "risk": [RISK_FOLDERS[r] for r in RISK_ORDER],
        "wmean_EUR_per_MWh": ys,
    }))
    plt.show()
    return means


TECH_GROUPS = [
    ("VRES", "GW", 1e-3, ("Gen_VRES_Solar", "Gen_VRES_Wind")),
    ("Electrolyzer", "MW", 1.0, ("Prod_H2_Green",)),
    ("Green ammonia", "MW", 1.0, ("Offtaker_Green",)),
]


def tech_value(case_key: str, risk: str, aids: tuple[str, ...], scale: float) -> float:
    return sum(inv_mw(case_key, risk, a) for a in aids) * scale


def plot_investment_case(case_key: str, *, stem: str):
    fig, axes = new_grid(1, 3, width=11.0, height=4.5, top=0.84, bottom=0.14, wspace=0.32)
    rows = []
    for ax, (name, unit, scale, aids) in zip(axes.ravel(), TECH_GROUPS):
        raw = [tech_value(case_key, rk, aids, scale) for rk in RISK_ORDER]
        vals = [v if _finite(v) and abs(v) >= 1e-6 else np.nan for v in raw]
        x = np.arange(len(RISK_ORDER))
        ax.bar(x, vals, color=[RISK_COLORS[r] for r in RISK_ORDER], width=0.62, zorder=3, linewidth=0)
        ax.set_xticks(x, risk_xlabels())
        finish_ax(ax)
        panel_unit(ax, f"{name}  ({unit} new)")
        finite = [v for v in vals if _finite(v)]
        hi = max(finite) if finite else 0.0
        ax.set_ylim(0, (hi * 1.18) if hi > 0 else 1.0)
        if hi <= 0:
            ax.text(
                0.5, 0.5, "No new capacity", transform=ax.transAxes,
                ha="center", va="center", fontproperties=FONT_SANS, fontsize=10, color=MIST,
            )
        rec = {"case": CASE_SHORT[case_key], "tech": name, "unit": unit}
        for rk, v in zip(RISK_ORDER, raw):
            rec[RISK_FOLDERS[rk]] = v
        rows.append(rec)
    add_chrome(fig, title=f"Green investment, {CASE_TITLES[case_key]}")
    save_figure(fig, stem, pd.DataFrame(rows))
    plt.show()


def commodity_metrics(case_key: str, risk: str) -> dict:
    ao = ao_full[case_key][risk]
    pr = prices[case_key][risk]
    wt = slot_weights(pr)
    n = len(pr)

    def col_or(name, fallback=None):
        if name in ao.columns:
            return ao[name].to_numpy(dtype=float)
        return fallback

    q_e = col_or("Cons_Elec_01_d")
    q_h = col_or("Prod_H2_Green_h2_out")
    if q_h is None:
        q_h = col_or("Offtaker_Green_h2_in")
    if len(D_EP_FLAT) == n:
        q_p = D_EP_FLAT
    else:
        q_p = ep_qty_vector(ao)

    p_e = pr["Elec_Price"].to_numpy(dtype=float)
    p_h = pr["H2_Price"].to_numpy(dtype=float)
    p_p = pr["EP_Price"].to_numpy(dtype=float)

    cwap_e = cwap_from_qty_price(q_e, p_e, wt) if q_e is not None else w_mean_price(pr, "Elec_Price")
    cwap_h = cwap_from_qty_price(q_h, p_h, wt) if q_h is not None else w_mean_price(pr, "H2_Price")
    cwap_p = cwap_from_qty_price(q_p, p_p, wt)

    def exp_qty(q, p):
        if q is None:
            return np.nan, np.nan
        qq = np.maximum(np.asarray(q, dtype=float), 0.0)
        pp = np.asarray(p, dtype=float)
        mask = np.isfinite(qq) & np.isfinite(pp) & np.isfinite(wt)
        return float(np.sum(wt[mask] * pp[mask] * qq[mask])), float(np.sum(wt[mask] * qq[mask]))

    e_exp, e_qty = exp_qty(q_e, p_e)
    h_exp, h_qty = exp_qty(q_h, p_h)
    p_exp, p_qty = exp_qty(q_p, p_p)
    return {
        "cwap_elec": cwap_e, "cwap_h2": cwap_h, "cwap_ep": cwap_p,
        "exp_elec": e_exp, "exp_h2": h_exp, "exp_ep": p_exp,
        "qty_elec": e_qty, "qty_h2": h_qty, "qty_ep": p_qty,
        "wmean_elec": w_mean_price(pr, "Elec_Price"),
        "wmean_h2": w_mean_price(pr, "H2_Price"),
        "wmean_ep": w_mean_price(pr, "EP_Price"),
    }


commodity: dict[str, pd.DataFrame] = {}
for ck in CASE_FOLDERS:
    rows = []
    for risk in RISK_ORDER:
        if risk not in prices.get(ck, {}):
            continue
        r = commodity_metrics(ck, risk)
        r.update(risk_folder=risk, risk_label=RISK_FOLDERS[risk])
        rows.append(r)
    if rows:
        commodity[ck] = pd.DataFrame(rows).set_index("risk_folder")

print("\nFigure helpers ready.")


### 1 — Green investment, social planner


In [ ]:
# 1. Green investment — social planner
if skip_missing_case("sp", "green investment"):
    pass
else:
    plot_investment_case("sp", stem="01_green_investment_sp")


### 2 — Green investment, market exposure


In [ ]:
# 2. Green investment — market exposure
if skip_missing_case("me", "green investment"):
    pass
else:
    plot_investment_case("me", stem="02_green_investment_me")


### 3 — Green investment, green social planner


In [ ]:
# 3. Green investment — green social planner
if skip_missing_case("gsp", "green investment"):
    pass
else:
    plot_investment_case("gsp", stem="03_green_investment_gsp")


### 4 — Green investment, green H2 social planner


In [ ]:
# 4. Green investment — green H2 social planner
if skip_missing_case("gh2", "green investment"):
    pass
else:
    plot_investment_case("gh2", stem="04_green_investment_gh2")


### 5 — Electricity prices, social planner


In [ ]:
# 5. SP electricity — duration curve and mean
plot_price_pair(
    "sp", "elec",
    title="Electricity prices, social planner",
    stem="05_sp_electricity_prices",
)


### 6 — Hydrogen prices, social planner


In [ ]:
# 6. SP hydrogen — duration curve and mean
plot_price_pair(
    "sp", "H2",
    title="Hydrogen prices, social planner",
    stem="06_sp_hydrogen_prices",
)


### 7 — Ammonia prices, social planner


In [ ]:
# 7. SP ammonia — duration curve and mean
plot_price_pair(
    "sp", "EP",
    title="Ammonia prices, social planner",
    stem="07_sp_ammonia_prices",
    unit="EUR / MWh_EP",
)


### 8 — Electricity prices, market exposure


In [ ]:
# 8. ME electricity — duration curve and mean
plot_price_pair(
    "me", "elec",
    title="Electricity prices, market exposure",
    stem="08_me_electricity_prices",
)


### 9 — Hydrogen prices, market exposure


In [ ]:
# 9. ME hydrogen — duration curve and mean
plot_price_pair(
    "me", "H2",
    title="Hydrogen prices, market exposure",
    stem="09_me_hydrogen_prices",
)


### 10 — Ammonia prices, market exposure


In [ ]:
# 10. ME ammonia — duration curve and mean
plot_price_pair(
    "me", "EP",
    title="Ammonia prices, market exposure",
    stem="10_me_ammonia_prices",
    unit="EUR / MWh_EP",
)


### 11 — Risk-adjusted cost of electricity, hydrogen and ammonia


In [ ]:
# 11. CWAP of electricity, hydrogen and ammonia — SP vs ME
pair = [ck for ck in ("sp", "me") if ck in commodity]
fig, axes = new_grid(1, 3, width=11.2, height=4.6, top=0.84, bottom=0.20, wspace=0.32)
markets = [
    ("cwap_elec", "Electricity", "EUR / MWh"),
    ("cwap_h2", "Hydrogen", "EUR / MWh"),
    ("cwap_ep", "Ammonia", "EUR / MWh_EP"),
]
x = np.arange(len(RISK_ORDER))
for ax, (col, name, unit) in zip(axes.ravel(), markets):
    for ck in pair:
        ys = series_for(commodity[ck], col)
        ax.plot(
            x, ys, color=CASE_COLORS[ck], lw=2.2, marker="o", markersize=6.2,
            markerfacecolor=PAPER, markeredgewidth=1.8, zorder=3, label=CASE_SHORT[ck],
        )
        for i, v in enumerate(ys):
            if not _finite(v):
                continue
            ax.text(x[i], v, f"  {v:.1f}", color=CASE_COLORS[ck], fontsize=7.4,
                    va="bottom", ha="left", fontproperties=FONT_SANS, path_effects=HALO)
    ax.set_xticks(x, risk_xlabels())
    finish_ax(ax)
    panel_unit(ax, f"{name}   ({unit})")
    ys_all = [v for ck in pair for v in series_for(commodity[ck], col)]
    nice_ylim(ax, ys_all, min_span=12.0 if col == "cwap_elec" else 8.0)

handles = [Line2D([0], [0], color=CASE_COLORS[ck], lw=2.2, marker="o", markerfacecolor=PAPER, markeredgewidth=1.6) for ck in pair]
fig_legend(fig, handles, [CASE_SHORT[ck] for ck in pair], ncol=2)
add_chrome(fig, title="Electricity, hydrogen and ammonia, SP and ME")
rows = []
for ck in pair:
    for rk in RISK_ORDER:
        if rk not in commodity[ck].index:
            continue
        r = commodity[ck].loc[rk]
        rows.append({
            "case": CASE_SHORT[ck], "risk": RISK_FOLDERS[rk],
            "elec_EUR_per_MWh": r["cwap_elec"], "h2_EUR_per_MWh": r["cwap_h2"],
            "ammonia_EUR_per_MWh": r["cwap_ep"],
        })
save_figure(fig, "11_sp_me_commodity_costs", pd.DataFrame(rows))
plt.show()


### 12 — CWAP across all four entry points


In [ ]:
# 12. CWAP — electricity, hydrogen, ammonia × all entry points
fig, axes = new_grid(1, 3, width=11.2, height=4.8, top=0.84, bottom=0.20, wspace=0.32)
markets = [
    ("cwap_elec", "Electricity", "EUR / MWh"),
    ("cwap_h2", "Hydrogen", "EUR / MWh"),
    ("cwap_ep", "Ammonia", "EUR / MWh_EP"),
]
x = np.arange(len(RISK_ORDER))
cases = [ck for ck in CASE_ORDER if ck in commodity]
for ax, (col, name, unit) in zip(axes.ravel(), markets):
    for ck in cases:
        ys = series_for(commodity[ck], col)
        ax.plot(
            x, ys, color=CASE_COLORS[ck], lw=2.15, marker="o", markersize=5.6,
            markerfacecolor=PAPER, markeredgewidth=1.6, zorder=3,
        )
    ax.set_xticks(x, risk_xlabels())
    finish_ax(ax)
    panel_unit(ax, f"{name}   ({unit})")
    ys_all = [v for ck in cases for v in series_for(commodity[ck], col)]
    nice_ylim(ax, ys_all, min_span=12.0 if col == "cwap_elec" else 8.0)

handles = [
    Line2D([0], [0], color=CASE_COLORS[ck], lw=2.15, marker="o",
           markerfacecolor=PAPER, markeredgewidth=1.5, label=CASE_SHORT[ck])
    for ck in cases
]
fig_legend(fig, handles, [CASE_SHORT[ck] for ck in cases], ncol=4)
add_chrome(fig, title="Consumer-weighted prices")
rows = []
for ck in cases:
    for rk in RISK_ORDER:
        if rk not in commodity[ck].index:
            continue
        r = commodity[ck].loc[rk]
        rows.append({
            "case": CASE_SHORT[ck], "risk": RISK_FOLDERS[rk],
            "elec_CWAP": r["cwap_elec"], "h2_CWAP": r["cwap_h2"], "ammonia_CWAP": r["cwap_ep"],
        })
save_figure(fig, "12_cwap_all_cases", pd.DataFrame(rows))
plt.show()


### 13 — Grey ammonia cost versus ME green ammonia


In [ ]:
# 13. Grey ammonia MC, then ME ammonia price per risk
if "me" not in commodity:
    print("Skipping grey/green ammonia gap — ME not loaded.")
else:
    me_ep = series_for(commodity["me"], "cwap_ep")
    labels = ["Grey\nexpected MC"] + risk_xlabels()
    values = [GREY_MC_EXPECTED] + me_ep
    colors = [REF] + [RISK_COLORS[r] for r in RISK_ORDER]
    rn = me_ep[0] if me_ep else np.nan
    hi_ra = me_ep[-1] if me_ep else np.nan

    fig, ax = new_figure(width=9.0, height=4.8)
    x = np.arange(len(labels))
    bars = ax.bar(x, values, color=colors, width=0.62, zorder=3, linewidth=0)
    ax.axhline(GREY_MC_EXPECTED, color=REF, ls=(0, (3, 2.5)), lw=1.05, zorder=2)
    for bar, val in zip(bars, values):
        if not _finite(val):
            continue
        ax.text(
            bar.get_x() + bar.get_width() / 2, val,
            f"{val:.1f}", ha="center", va="bottom", fontsize=8.5,
            color=INK, fontproperties=FONT_SANS_SB, clip_on=False,
        )
    ax.set_xticks(x, labels)
    finish_ax(ax)
    panel_unit(ax, "EUR / MWh_EP")
    finite = [v for v in values if _finite(v)]
    ax.set_ylim(0, max(finite) * 1.22)

    add_chrome(fig, title="Grey ammonia cost and ME ammonia prices")
    save_figure(fig, "13_me_ammonia_grey_gap", pd.DataFrame({
        "series": ["Grey expected MC"] + [RISK_FOLDERS[r] for r in RISK_ORDER],
        "EUR_per_MWh_EP": values,
        "gap_vs_grey": [v - GREY_MC_EXPECTED if _finite(v) else np.nan for v in values],
    }))
    plt.show()
